# Módulo 4 · Formatos: CSV, Parquet y Delta

⏱ **23 min** · Slides: 24–26 (por filas vs por columnas, comparativo, Delta = Parquet + log) + mantenimiento Delta

**Objetivo:** guardar *los mismos datos* en tres formatos y medir las diferencias.

| Paso | Qué demuestras |
|---|---|
| 1 | Preparar un DataFrame plano |
| 2 | Escribir en CSV, Parquet y Delta |
| 3 | **Tamaño**: ¿cuál pesa menos? |
| 4 | **Esquema**: ¿quién recuerda los tipos? |
| 5 | **Columnar**: ¿qué columnas se leen? |
| 6 | **Delta = Parquet + `_delta_log`** |
| 7 | **ACID**: una escritura fallida no daña la tabla |
| 8 | **Time travel**: volver a una versión anterior |
| 9 | **Archivos pequeños**: el problema que motiva `OPTIMIZE` |
| 10 | **`OPTIMIZE`**: compactar |
| 11 | **`ZORDER BY`**: agrupar por columna y saltar archivos |
| 12 | **`VACUUM`**: borrar archivos viejos (y su costo) |

**Requiere:** módulo 2.

In [0]:
%run ./00_Setup

## Paso 1 · Preparar un DataFrame plano

CSV no soporta estructuras anidadas ni listas, así que aplanamos el JSON a columnas simples.

In [0]:
plano = read_raw().select(
    F.col("id").alias("episode_id"),
    F.col("name").alias("episode_name"),
    "season",
    F.col("number").alias("episode_number"),
    F.col("runtime").alias("runtime_min"),
    F.col("rating.average").alias("episode_rating"),
    "airdate",
    F.col("_embedded.show.id").alias("show_id"),
    F.col("_embedded.show.name").alias("show_name"),
    F.col("_embedded.show.language").alias("show_language"),
    F.concat_ws(",", F.col("_embedded.show.genres")).alias("show_genres"),   # lista → texto
    F.col("_embedded.show.summary").alias("show_summary"),
)

print("Filas:", plano.count())
display(plano.limit(5))

## Paso 2 · Escribir en los tres formatos

In [0]:
csv_path     = f"{FORMATOS_PATH}/csv"
parquet_path = f"{FORMATOS_PATH}/parquet"
delta_path   = f"{FORMATOS_PATH}/delta"

plano.write.mode("overwrite").option("header", True).csv(csv_path)
plano.write.mode("overwrite").parquet(parquet_path)
plano.write.mode("overwrite").format("delta").save(delta_path)

print("✔ Tres copias escritas en", FORMATOS_PATH)

## Paso 3 · Tamaño en disco

In [0]:
tamanos = [("CSV", size_mb(csv_path)), ("Parquet", size_mb(parquet_path)), ("Delta", size_mb(delta_path))]
display(spark.createDataFrame(tamanos, "formato STRING, mb DOUBLE"))

👀 **Qué observar:** los mismos datos pesan mucho menos en Parquet (binario, comprimido).
Delta pesa casi igual que Parquet: por dentro **son archivos Parquet** (más un pequeño log).
Pista: usa el selector de gráfico de barras sobre la tabla.

## Paso 4 · Esquema: ¿quién recuerda los tipos?

In [0]:
print("CSV sin esquema → todo texto:")
spark.read.option("header", True).csv(csv_path).select("episode_id", "runtime_min", "airdate").printSchema()

In [0]:
print("Parquet → tipos incluidos en el archivo:")
spark.read.parquet(parquet_path).select("episode_id", "runtime_min", "airdate").printSchema()

## Paso 5 · Delta = Parquet + un log de transacciones

In [0]:
display(dbutils.fs.ls(delta_path))

In [0]:
display(dbutils.fs.ls(f"{delta_path}/_delta_log"))

👀 **Qué observar:** la carpeta tiene archivos `.parquet` y una carpeta `_delta_log` con JSON.
Una tabla Delta **es lo que dice el log**: un archivo Parquet que no aparece en el log, no existe para la tabla.
Ese log es lo que da transacciones y versiones.

## Paso 7 · ACID: todo o nada

Simulamos un job que **falla a mitad de una escritura** (`overwrite`) sobre la tabla Delta.

In [0]:
from delta.tables import DeltaTable

tabla = DeltaTable.forPath(spark, delta_path)

def current_state():
    filas = spark.read.format("delta").load(delta_path).count()
    version = tabla.history().agg(F.max("version")).first()[0]
    return filas, version

filas_antes, version_antes = current_state()
print(f"ANTES   → filas: {filas_antes} | versión: {version_antes}")

In [0]:
# raise_error() hace fallar la escritura cuando llega a ciertas filas
roto = plano.withColumn(
    "runtime_min",
    F.when(F.col("episode_id") % 20 == 0, F.expr("raise_error('Fallo simulado a mitad de la escritura')"))
     .otherwise(F.col("runtime_min")))

try:
    roto.write.mode("overwrite").format("delta").save(delta_path)
except Exception as e:
    print("❌ La escritura falló:", str(e).splitlines()[0][:150])

In [0]:
filas_despues, version_despues = current_state()
print(f"DESPUÉS → filas: {filas_despues} | versión: {version_despues}")
print("¿Tabla intacta? ", (filas_antes, version_antes) == (filas_despues, version_despues))

👀 **Qué observar:** a pesar del error, la tabla quedó **exactamente igual**: mismas filas, misma versión.
El commit nunca se registró en el log, así que nadie ve una escritura a medias.

## Paso 8 · Time travel: volver atrás

Cada cambio exitoso crea una versión nueva. Borremos filas "por error" y recuperémoslas.

In [0]:
tabla.delete("episode_rating IS NULL")            # "error": borra todo episodio sin calificación
print("Filas tras el DELETE:", spark.read.format("delta").load(delta_path).count())

In [0]:
display(tabla.history().select("version", "operation", "operationMetrics"))

In [0]:
# Leer la versión 0 (antes del DELETE)
v0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)

v1 = spark.read.format("delta").load(delta_path)
print("Filas en la versión 0:", v0.count())
print("Filas en la versión 1:", v1.count())

In [0]:
tabla.restoreToVersion(0)
print("Filas tras RESTORE a la versión 0:", spark.read.format("delta").load(delta_path).count())

👀 **Qué observar:** `DELETE` creó la versión 1; `RESTORE` creó la 2 y devolvió los datos.
Con Parquet o CSV esto no es posible: los archivos son inmutables y no hay historial.

# Mantenimiento de tablas Delta: OPTIMIZE, Z-ORDER y VACUUM

Una tabla Delta que recibe muchas escrituras se **fragmenta** y **acumula archivos viejos**. Tres comandos la mantienen sana:

| Comando | Problema que resuelve | Evidencia que mediremos |
|---|---|---|
| `OPTIMIZE` | Muchos archivos pequeños (lecturas lentas) | Número de archivos antes y después |
| `OPTIMIZE ... ZORDER BY` | Filas de una misma clave dispersas en todos los archivos | En cuántos archivos está una misma serie |
| `VACUUM` | Archivos viejos que ya nadie usa y ocupan espacio | Archivos físicos vs archivos de la tabla |

Trabajamos con una **copia nueva** (`delta_mantenimiento`) para no alterar la tabla de los pasos anteriores.

## Paso 9 · El problema: archivos pequeños

Una carga en streaming o muchas cargas pequeñas dejan decenas de archivos diminutos.
Lo simulamos escribiendo los mismos datos repartidos en **40 archivos**.

Primero, dos funciones de medición (se usarán en todos los pasos siguientes).

In [0]:
dbutils.fs.rm(mant_path, recurse=True)

In [0]:
mant_path = f"{FORMATOS_PATH}/delta_mantenimiento"

def table_files(path):
    """Archivos que pertenecen a la tabla (según el log) y su tamaño en KB."""
    d = spark.sql(f"DESCRIBE DETAIL delta.`{path}`").first()
    return d["numFiles"], round(d["sizeInBytes"] / 1024)

def physical_files(path):
    """Archivos .parquet que hay realmente en la carpeta (incluye los viejos)."""
    return len([f for f in dbutils.fs.ls(path) if f.name.endswith(".parquet")])

def files_with_show(path, show_id):
    """En cuántos archivos hay al menos una fila de esa serie."""
    return (spark.read.format("delta").load(path)
            .where(F.col("show_id") == show_id)
            .select("_metadata.file_path").distinct().count())

In [0]:
# 40 archivos con filas repartidas al azar: cada archivo mezcla series de todo tipo
plano.repartition(40).write.mode("overwrite").format("delta").save(mant_path)

n_archivos, kb = table_files(mant_path)
print(f"Archivos: {n_archivos} | Tamaño total: {kb} KB | Promedio: {kb // n_archivos} KB por archivo")

In [0]:
# La serie con más episodios del periodo: nuestra "consulta de prueba" para todo el módulo
fila = plano.groupBy("show_id", "show_name").count().orderBy(F.col("count").desc()).first()
serie_id, serie_nombre, serie_filas = fila["show_id"], fila["show_name"], fila["count"]

print(f"Serie de prueba: {serie_nombre} (id {serie_id}) con {serie_filas} episodios")
print(f"Archivos que hay que abrir para encontrarla: {files_with_show(mant_path, serie_id)} de {n_archivos}")

👀 **Qué observar:** para encontrar una sola serie hay que abrir casi **todos** los archivos, porque cada uno tiene un poco de todo.
Abrir 40 archivos diminutos es mucho más lento que abrir pocos archivos bien llenos.

## Paso 10 · `OPTIMIZE`: compactar archivos pequeños

`OPTIMIZE` junta muchos archivos pequeños en pocos grandes. El tamaño objetivo normal es ~1 GB;

In [0]:
spark.sql(f"ALTER TABLE delta.`{mant_path}` SET TBLPROPERTIES ('delta.targetFileSize' = '128kb')")

resultado = spark.sql(f"OPTIMIZE delta.`{mant_path}`")
display(resultado.select("metrics.numFilesRemoved", "metrics.numFilesAdded"))

In [0]:
n_archivos, kb = table_files(mant_path)
print(f"Archivos ahora: {n_archivos} | Tamaño total: {kb} KB | Promedio: {kb // n_archivos} KB por archivo")
print(f"Archivos que hay que abrir para la serie '{serie_nombre}': {files_with_show(mant_path, serie_id)} de {n_archivos}")

👀 **Qué observar:** de 40 archivos pasamos a unos pocos, más grandes (los datos son los mismos).
Pero ojo: para encontrar la serie **seguimos abriendo casi todos**. `OPTIMIZE` junta archivos, **no ordena** las filas.
Eso lo resuelve el siguiente paso.

## Paso 11 · `ZORDER BY`: agrupar filas parecidas

`OPTIMIZE ... ZORDER BY (columna)` además **reordena** las filas para que las de una misma clave queden juntas en pocos archivos.
Delta guarda el mínimo y máximo de cada columna por archivo, así que una consulta con filtro `WHERE show_id = X`
**salta** (*data skipping*) los archivos cuyo rango no incluye X.

In [0]:
resultado = spark.sql(f"OPTIMIZE delta.`{mant_path}` ZORDER BY (show_id)")
display(resultado.select("metrics.numFilesRemoved", "metrics.numFilesAdded"))

In [0]:
n_archivos, kb = table_files(mant_path)
print(f"Archivos ahora: {n_archivos} | Tamaño total: {kb} KB")
print(f"Archivos que hay que abrir para la serie '{serie_nombre}': {files_with_show(mant_path, serie_id)} de {n_archivos}")

In [0]:
# La consulta de negocio devuelve exactamente lo mismo, pero ahora Delta puede saltarse casi todos los archivos
print("Episodios de la serie:", spark.read.format("delta").load(mant_path).where(F.col("show_id") == serie_id).count(),
      "| esperados:", serie_filas)

👀 **Qué observar:** las filas de la serie pasaron de estar esparcidas en casi todos los archivos a estar en **uno o dos**.
El resultado de la consulta es idéntico; lo que cambia es cuántos archivos hay que leer.

| Cuándo | Recomendación |
|---|---|
| Columnas por las que más filtras (`WHERE`) o unes (`JOIN`) | Úsalas en `ZORDER BY` (pocas: 1 a 3) |
| Columnas de muy baja cardinalidad (sí/no) | No sirven para Z-ORDER |
| Tablas nuevas | Databricks recomienda *liquid clustering* (`CLUSTER BY`), que automatiza esto; Z-ORDER es el concepto base |

## Paso 12 · `VACUUM`: borrar archivos que ya no se usan

`OPTIMIZE` **no borra** los archivos viejos: solo los marca como eliminados en el log (por eso el time travel sigue funcionando).
Siguen ocupando espacio hasta que ejecutes `VACUUM`.

In [0]:
n_tabla, _ = table_files(mant_path)
n_fisicos = physical_files(mant_path)
print(f"Archivos que usa la tabla : {n_tabla}")
print(f"Archivos en la carpeta    : {n_fisicos}   ← el resto son archivos viejos de 40 + compactados")

**12.1 · La protección de Delta.** Por defecto `VACUUM` solo borra archivos con más de **7 días** de antigüedad.
Intentar borrar todo ahora mismo se bloquea:

In [0]:
try:
    spark.sql(f"VACUUM delta.`{mant_path}` RETAIN 0 HOURS")
except Exception as e:
    print("🛡 Delta lo bloqueó:", str(e).splitlines()[0][:200])

**12.2 · Antes de borrar, el time travel todavía funciona.** La versión 0 es la tabla de 40 archivos:

In [0]:
v0 = spark.read.format("delta").option("versionAsOf", 0).load(mant_path)
print("Filas en la versión 0 (antes de OPTIMIZE):", v0.count())

**12.3 · `DRY RUN`: ver qué se borraría sin borrar nada.** Solo para el laboratorio bajamos la retención a 0 horas.

> ⚠️ **En producción nunca uses 0 horas.** Rompe el time travel y puede romper lecturas que estén en curso.
> Lo habitual es dejar 7 días o más.

In [0]:
spark.sql(f"ALTER TABLE delta.`{mant_path}` SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')")

a_borrar = spark.sql(f"VACUUM delta.`{mant_path}` RETAIN 0 HOURS DRY RUN")
print("Archivos que se borrarían:", a_borrar.count())
display(a_borrar.limit(5))

**12.4 · Ejecutar `VACUUM`.**

In [0]:
spark.sql(f"VACUUM delta.`{mant_path}` RETAIN 0 HOURS")

n_tabla, kb = table_files(mant_path)
print(f"Archivos que usa la tabla : {n_tabla}")
print(f"Archivos en la carpeta    : {physical_files(mant_path)}   ← ya coinciden")

**12.5 · El precio: se pierde el time travel anterior al `VACUUM`.**

In [0]:
try:
    spark.read.format("delta").option("versionAsOf", 0).load(mant_path).count()
    print("La versión 0 todavía se puede leer")
except Exception as e:
    print("❌ La versión 0 ya no existe:", str(e).splitlines()[0][:160])

👀 **Qué observar:** `VACUUM` dejó en la carpeta solo los archivos de la tabla y **liberó espacio**,
pero las versiones antiguas dejaron de poder leerse. Por eso existe el periodo de retención: es tu ventana de recuperación.

| | Qué hace | ¿Borra datos? | ¿Afecta el time travel? |
|---|---|---|---|
| `OPTIMIZE` | Compacta archivos | No (los viejos quedan) | No |
| `ZORDER BY` | Compacta y agrupa por columna | No | No |
| `VACUUM` | Borra archivos viejos | Solo archivos ya descartados | **Sí**: pierdes versiones antiguas |

**Orden habitual:** `OPTIMIZE` (con `ZORDER BY`) periódicamente → `VACUUM` después, con retención de 7 días o más.

## Resumen del módulo

| | CSV | Parquet | Delta |
|---|---|---|---|
| Tamaño | Mayor | Menor | Menor |
| Esquema | No guarda tipos | Sí | Sí, y lo valida |
| Lectura columnar | No | Sí | Sí |
| ACID | No | No | **Sí** |
| Time travel | No | No | **Sí** |
| Mantenimiento (`OPTIMIZE`, `ZORDER`, `VACUUM`) | No | No | **Sí** |

## ✔ Checkpoint

- [ ] Medí que Parquet/Delta pesan menos que CSV
- [ ] Vi `_delta_log` y sé que Delta = Parquet + log
- [ ] Comprobé que una escritura fallida no altera la tabla
- [ ] Recuperé datos con `versionAsOf` y `RESTORE`
- [ ] Compacté 40 archivos con `OPTIMIZE` y vi que **no** ordena las filas
- [ ] Con `ZORDER BY` redujeron los archivos a abrir para una misma serie
- [ ] Con `VACUUM` liberé espacio y entendí que se pierde el time travel antiguo

**Siguiente:** `05_Bronze_multiples_archivos`

In [0]:
dbutils.fs.rm(mant_path, recurse=True)